# Kimi-Linear (GDN-2) code LM — train & evaluate on Google Colab (T4 GPU)

This notebook runs the full pipeline in this repo — a decoder-only **Kimi-Linear**
language model with a **Gated DeltaNet-2** linear-attention backbone — trained on
**OpenCoder** datasets using **Grain** (data), **Optax** (optimizer), and
**Orbax** (checkpoints).

**Before you start:** enable the GPU runtime.
> `Runtime` → `Change runtime type` → **Hardware accelerator: T4 GPU** → `Save`.

The steps below:
1. Check the GPU 
2. Install JAX (CUDA 12) + deps 
3. Get the project code
4. (optional) HF login
5. Smoke test
6. Write a T4-tuned config
7. Pretrain
8. Evaluate & generate
9. (optional) Persist to Drive
10. (optional) Instruction-tune + HumanEval.

## 1. Check the GPU

You should see a **Tesla T4** with ~15 GB of memory. If this errors or shows no
GPU, set the runtime to **T4 GPU** (see above) and re-run.

In [ ]:
!nvidia-smi

## 2. Install JAX (CUDA 12) + dependencies

This installs a CUDA-enabled JAX plus the pipeline's libraries (Flax, Optax,
Orbax, Grain, Datasets, Tokenizers). It takes a minute or two.

> **If step 2b below does NOT list a `cuda`/`gpu` device**, do
> `Runtime` → `Restart session`, then re-run **from step 2b** (skip this install
> cell). Reinstalling JAX sometimes needs a fresh kernel to load the CUDA plugin.

In [ ]:
# CUDA-enabled JAX for Colab's CUDA 12 runtime + the pipeline dependencies.
!pip install -q -U "jax[cuda12]" flax optax "orbax-checkpoint>=0.11" grain \
    "datasets>=4.0" "tokenizers>=0.20" huggingface_hub pyyaml

### 2b. Verify JAX sees the T4

In [ ]:
import jax
print("jax", jax.__version__)
print("devices:", jax.devices())
assert jax.default_backend() == "gpu", (
    "No GPU backend. Set runtime to T4 GPU, then Runtime > Restart session and "
    "re-run from step 2b."
)
print("OK - running on", jax.devices()[0].device_kind)

## 3. Get the project code

Pick **one** setup mode by editing `SETUP_MODE` below:

- `"git"` — clone your GitHub repo (set `REPO_URL`). Easiest if you've pushed the
  project to GitHub.
- `"upload"` — upload a `.zip` of the project from your computer (use the file
  picker that appears).
- `"drive"` — the project already lives in your Google Drive (set `DRIVE_PATH`).

In [ ]:
import os

SETUP_MODE = "git"                       # "git" | "upload" | "drive"
REPO_URL   = ""                          # e.g. "https://github.com/you/your-repo.git"
DRIVE_PATH = "/content/drive/MyDrive/nugie-coding-llm-agent-3"  # for SETUP_MODE="drive"

PROJECT_DIR = "/content/project"

if SETUP_MODE == "git":
    assert REPO_URL, "Set REPO_URL to your repository URL."
    !rm -rf "$PROJECT_DIR"
    !git clone "$REPO_URL" "$PROJECT_DIR"

elif SETUP_MODE == "upload":
    from google.colab import files
    import zipfile, glob, shutil
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Select a .zip of the project (must contain kimi_linear_gdn2.py + training/).")
    up = files.upload()
    zname = next(n for n in up if n.endswith(".zip"))
    with zipfile.ZipFile(zname) as z:
        z.extractall("/content/_unzip")
    # find the folder that actually contains the project and move it into place
    root = next(p for p in glob.glob("/content/_unzip/**/kimi_linear_gdn2.py", recursive=True))
    src = os.path.dirname(root)
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    shutil.move(src, PROJECT_DIR)

elif SETUP_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PATH

else:
    raise ValueError(SETUP_MODE)

os.chdir(PROJECT_DIR)
print("Project dir:", os.getcwd())
assert os.path.exists("kimi_linear_gdn2.py") and os.path.isdir("training"), \
    "Project files not found - check SETUP_MODE / paths."
print("Contents:", sorted(os.listdir(".")))

## 4. (optional) HuggingFace login

OpenCoder datasets are public, so this is optional — but a token gives higher rate
limits and faster streaming downloads. Skip if you don't have one.

In [ ]:
# from huggingface_hub import login
# login()   # paste an access token from https://huggingface.co/settings/tokens

## 5. Smoke test (offline, ~1 min)

Runs the whole pipeline on synthetic data with no download — a fast pre-flight that
the tokenizer, Grain data path, model, Optax step, and Orbax checkpointing all work
on this GPU.

In [ ]:
!python -m tests.smoke_test

## 6. Write a T4-tuned training config

A ~140M-parameter model (d_model 512, 8 layers, 8-expert MoE) at seq_len 512,
batch 8 — sized to train comfortably within the T4's ~15 GB.

**Memory / speed knobs** if you hit OOM or want to go bigger:
- `compute_dtype: bfloat16` roughly halves activation memory (the numerically
  sensitive parts stay fp32 regardless). float32 is the safe default here.
- Lower `train.batch_size` or `data.seq_len` to reduce memory.
- Raise `model.d_model` / `n_layers` / `moe_n_routed` and `train.total_steps` to
  scale up.

In [ ]:
%%writefile configs/colab_pretrain.yaml
model:
  vocab_size: 16384          # target BPE vocab (overwritten by the trained tokenizer)
  d_model: 512
  n_layers: 8
  full_attn_period: 4        # 3:1 linear:full (MLA on layers 3, 7)
  gdn_num_heads: 8
  gdn_head_k_dim: 64
  gdn_head_v_dim: 64
  gdn_chunk_size: 64         # seq_len must be a multiple of this
  gdn_conv_size: 4
  mla_num_q_heads: 8
  mla_num_kv_heads: 2
  mla_head_dim: 64
  max_seq_len: 1024
  moe_d_ff: 1024
  moe_n_routed: 8
  moe_n_shared: 1
  moe_top_k: 2
  compute_dtype: float32     # T4-safe; set bfloat16 to halve activation memory

data:
  task: pretrain
  sources:
    - repo: OpenCoder-LLM/opc-annealing-corpus
      name: algorithmic_corpus
      split: train
      max_docs: 40000
    - repo: OpenCoder-LLM/opc-annealing-corpus
      name: synthetic_code_snippet
      split: train
      max_docs: 40000
  text_field: text
  seq_len: 512
  val_fraction: 0.01
  max_val_docs: 256
  streaming: true

optim:
  peak_lr: 3.0e-4
  min_lr_ratio: 0.1
  warmup_steps: 100
  weight_decay: 0.1
  grad_clip: 1.0
  router_bias_lr: 1.0e-3

train:
  batch_size: 8
  total_steps: 2000
  seed: 0
  log_every: 20
  eval_every: 250
  eval_batches: 20
  save_every: 500
  keep_checkpoints: 3
  out_dir: /content/runs/colab_pretrain
  tokenizer_path: /content/runs/colab_pretrain/tokenizer.json

## 7. Pretrain

First run trains a ByteLevel-BPE tokenizer on the corpus, materializes the packed
dataset, then trains. Watch the `ce` / `ppl` columns fall and the `VAL ppl` drop at
each eval. You can stop anytime — the last checkpoint under `out_dir` is resumable.

> **Tip:** to continue a stopped run, re-run this cell with `--resume` appended.

In [ ]:
!python -m training.train --config configs/colab_pretrain.yaml
# Resume an interrupted run instead:
# !python -m training.train --config configs/colab_pretrain.yaml --resume

## 8. Evaluate & generate

Reports held-out perplexity and greedy code completions from a few prompts.

In [ ]:
!python -m training.evaluate --config configs/colab_pretrain.yaml --max-new-tokens 96

## 9. (optional) Persist checkpoints to Google Drive

Colab runtimes are ephemeral — copy the run to Drive so it survives disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/kimi_runs"
!cp -r /content/runs/colab_pretrain "/content/drive/MyDrive/kimi_runs/"
print("Saved to Drive: MyDrive/kimi_runs/colab_pretrain")
# To resume later from Drive, copy it back and run training with --resume:
# !cp -r "/content/drive/MyDrive/kimi_runs/colab_pretrain" /content/runs/

## 10. (optional) Instruction-tune (SFT) + HumanEval

Fine-tune on OpenCoder's `(instruction, output)` pairs with a **prompt-masked loss**
(only the answer is supervised). This **warm-starts from the pretrained checkpoint**
via `--init-from` (weights only; fresh optimizer + data), and reuses the pretraining
tokenizer so the vocab matches — so the two stages actually chain. With a modest step
budget HumanEval pass@1 will still be low, but the full evaluation path is runnable.

> ⚠️ `--humaneval` **executes model-generated code** in a subprocess. Colab is a
> disposable VM, which is an appropriate place to run it.

In [ ]:
%%writefile configs/colab_sft.yaml
model:
  vocab_size: 16384
  d_model: 512
  n_layers: 8
  full_attn_period: 4
  gdn_num_heads: 8
  gdn_head_k_dim: 64
  gdn_head_v_dim: 64
  gdn_chunk_size: 64
  gdn_conv_size: 4
  mla_num_q_heads: 8
  mla_num_kv_heads: 2
  mla_head_dim: 64
  max_seq_len: 1024
  moe_d_ff: 1024
  moe_n_routed: 8
  moe_n_shared: 1
  moe_top_k: 2
  compute_dtype: float32

data:
  task: sft
  sources:
    - repo: OpenCoder-LLM/opc-sft-stage1
      name: realuser_instruct
      split: train
      max_docs: 20000
    - repo: OpenCoder-LLM/opc-sft-stage2
      name: evol_instruct
      split: train
      max_docs: 20000
  instruction_field: instruction
  response_field: output
  seq_len: 512
  val_fraction: 0.02
  max_val_docs: 256
  streaming: true

optim:
  peak_lr: 1.0e-4
  min_lr_ratio: 0.1
  warmup_steps: 100
  weight_decay: 0.1
  grad_clip: 1.0
  router_bias_lr: 1.0e-3

train:
  batch_size: 8
  total_steps: 1500
  seed: 0
  log_every: 20
  eval_every: 250
  eval_batches: 20
  save_every: 500
  keep_checkpoints: 3
  out_dir: /content/runs/colab_sft
  tokenizer_path: /content/runs/colab_pretrain/tokenizer.json

In [ ]:
# Warm-start SFT from the pretrained checkpoint (weights only; fresh optimizer/data).
!python -m training.train --config configs/colab_sft.yaml \
    --init-from /content/runs/colab_pretrain

In [ ]:
!python -m training.evaluate --config configs/colab_sft.yaml \
    --humaneval --humaneval-limit 20 --max-new-tokens 256

## Troubleshooting

- **No GPU / `default_backend()` is `cpu`** — set runtime to T4, then
  `Runtime` → `Restart session` and re-run from step 2b.
- **`ResourceExhaustedError` (OOM)** — lower `train.batch_size` or `data.seq_len`,
  or set `compute_dtype: bfloat16` in the config, then re-run.
- **Slow dataset build** — reduce `max_docs` in the config; the first run also
  trains the tokenizer, which is a one-time cost per `tokenizer_path`.
- **Disconnected** — re-run steps 1-3, then training with `--resume` (after copying
  the run back from Drive if you saved it in step 9).